# Tech Challenge — Fase 2
## Classificando a Qualidade de Vinhos com Machine Learning

**Base de dados:** [Wine Quality Dataset (Kaggle)](https://www.kaggle.com/datasets/yasserh/wine-quality-dataset) — variante tinta do "Vinho Verde" português (1.143 amostras, 11 variáveis físico-químicas).

**Objetivo:** treinar e comparar modelos de classificação capazes de prever se um vinho é de **Alta Qualidade** (nota ≥ 7) ou **Baixa/Média Qualidade** (nota < 7), a partir de suas características físico-químicas.

Este notebook segue a pipeline completa do desafio:
1. Compreensão do problema
2. Análise Exploratória de Dados (EDA)
3. Pré-processamento de dados
4. Desenvolvimento de modelos (Random Forest e SVM)
5. Avaliação dos modelos
6. Interpretação dos resultados


## 0. Setup

In [ ]:
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42


## 1. Compreensão do Problema

A indústria vitivinícola depende tradicionalmente de avaliação sensorial feita por especialistas — um processo caro, demorado e sujeito a subjetividade. A proposta aqui é usar variáveis físico-químicas, medidas objetivamente durante a produção, para **prever a categoria de qualidade do vinho**, apoiando decisões de produção e padronização.

**Variável alvo:** transformamos a nota de qualidade original (escala 3–8 neste dataset) em uma variável binária:

- `1` (Alta Qualidade): nota ≥ 7
- `0` (Baixa/Média Qualidade): nota < 7

Essa simplificação torna o problema mais direto de interpretar e de comunicar para um público não técnico (ex: "esse lote tem alta probabilidade de ser um vinho premium"), além de ser a abordagem sugerida no desafio.

In [ ]:
df = pd.read_csv("../data/WineQT.csv")
df = df.drop(columns=["Id"])  # identificador da amostra, sem valor preditivo
df.head()


In [ ]:
print("Dimensões:", df.shape)
df.info()


In [ ]:
df.describe().T


## 2. Análise Exploratória de Dados (EDA)

### 2.1 Valores faltantes e duplicados

In [ ]:
print("Valores faltantes por coluna:")
print(df.isnull().sum())
print("\nTotal de valores faltantes:", df.isnull().sum().sum())


**Não há valores faltantes** no dataset — não é necessário nenhum tratamento de imputação.

Ainda assim, vale checar linhas duplicadas: como as variáveis físico-químicas são medidas numéricas arredondadas, é comum que **duas amostras distintas de vinho apresentem exatamente os mesmos valores** medidos, sem que isso signifique erro de coleta.

In [ ]:
dup_rows = df.duplicated().sum()
print(f"Linhas com todas as variáveis (incl. nota) idênticas: {dup_rows} ({100*dup_rows/len(df):.1f}%)")


**Decisão:** mantemos as linhas duplicadas. Elas representam **amostras de vinho diferentes que, por limitação de precisão das medições físico-químicas, coincidem em valores** — não são erros de duplicação de registro (cada linha do dataset original tinha um `Id` único, que removemos por não ser preditivo, mas que confirma se tratarem de amostras distintas). Remover essas linhas reduziria artificialmente o já pequeno volume de dados e poderia enviesar a proporção de vinhos de alta qualidade, que já é minoritária.

### 2.2 Distribuição das variáveis

In [ ]:
feature_cols = [c for c in df.columns if c != "quality"]

fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()
for i, col in enumerate(feature_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="#7B2D26")
    axes[i].set_title(col, fontsize=11)
axes[-1].axis("off")
plt.tight_layout()
plt.savefig("../results/01_distribuicoes.png", bbox_inches="tight")
plt.show()


**Leitura:** a maioria das variáveis apresenta forte assimetria à direita (ex: `residual sugar`, `chlorides`, `total sulfur dioxide`), com uma cauda longa de valores altos — típico de processos físico-químicos naturais, mas também um indício de outliers a investigar. `pH` e `density` são as mais próximas de uma distribuição normal, o que é esperado por serem variáveis fisicamente mais restritas.

### 2.3 Outliers

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()
for i, col in enumerate(feature_cols):
    sns.boxplot(y=df[col], ax=axes[i], color="#C9A227")
    axes[i].set_title(col, fontsize=11)
axes[-1].axis("off")
plt.tight_layout()
plt.savefig("../results/02_boxplots_outliers.png", bbox_inches="tight")
plt.show()


In [ ]:
outlier_summary = {}
for col in feature_cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[col] < low) | (df[col] > high)).sum())
    outlier_summary[col] = {"n_outliers": n_out, "pct": round(100 * n_out / len(df), 2)}

pd.DataFrame(outlier_summary).T.sort_values("pct", ascending=False)


**Leitura:** usando o método do Intervalo Interquartil (IQR — valores fora de `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`), variáveis como `residual sugar`, `chlorides` e `sulphates` concentram a maior proporção de outliers (entre 3% e 10% das amostras). Isso é consistente com o processo de vinificação: açúcar residual e cloretos variam bastante entre lotes e safras, e amostras "extremas" nessas variáveis não são necessariamente erros — podem ser justamente vinhos atípicos, que inclusive podem estar entre os de nota mais alta ou mais baixa. Por isso o tratamento (feito na seção de pré-processamento) será por **capping (winsorização)**, não remoção — ver justificativa detalhada adiante.

### 2.4 Correlação entre variáveis

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Matriz de correlação")
plt.tight_layout()
plt.savefig("../results/03_correlacao.png", bbox_inches="tight")
plt.show()


In [ ]:
corr["quality"].drop("quality").sort_values(ascending=False)


**Leitura das correlações mais relevantes com `quality`:**

- **`alcohol` (+0.48):** a correlação mais forte do dataset. Vinhos com maior teor alcoólico tendem a ser mais bem avaliados — fisiologicamente, mais álcool costuma indicar uvas mais maduras na colheita, associadas a perfis sensoriais mais ricos.
- **`volatile acidity` (-0.41):** segunda correlação mais forte, e negativa. Acidez volátil elevada está associada ao gosto de vinagre (ácido acético), um defeito sensorial clássico — faz total sentido que penalize a nota.
- **`sulphates` (+0.26)** e **`citric acid` (+0.24):** sulfatos atuam como antimicrobiano/antioxidante (conservação), e ácido cítrico contribui para frescor — ambos coerentes com vinhos de melhor qualidade percebida.
- **`total sulfur dioxide` (-0.18)** e **`density` (-0.18):** correlações mais fracas, mas negativas — excesso de SO2 total pode indicar conservantes em excesso (alterando aroma), e maior densidade tende a acompanhar menor teor alcoólico.
- Variáveis como `residual sugar`, `pH` e `free sulfur dioxide` têm correlação linear muito fraca com a nota — não significa que sejam irrelevantes (podem ter efeitos não lineares ou de interação, capturáveis por modelos como Random Forest), mas isoladamente explicam pouco.

Também vale checar multicolinearidade entre preditores: `fixed acidity` correlaciona fortemente com `citric acid` (+0.67) e `density` (+0.67), e `free sulfur dioxide` com `total sulfur dioxide` (+0.67) — esperado, já que são quimicamente relacionados. Modelos baseados em árvores (Random Forest) lidam bem com isso; para o SVM, o escalonamento ajuda a mitigar o impacto.

### 2.5 Transformação da variável alvo e balanceamento das classes

In [ ]:
df["high_quality"] = (df["quality"] >= 7).astype(int)

class_counts = df["high_quality"].value_counts().sort_index()
class_pct = (class_counts / len(df) * 100).round(2)
print(class_counts)
print(class_pct)

plt.figure(figsize=(6, 4.5))
ax = sns.countplot(x="high_quality", data=df, palette=["#7B2D26", "#C9A227"])
ax.set_xticklabels(["Baixa/Média (<7)", "Alta (>=7)"])
ax.set_xlabel("")
ax.set_ylabel("Nº de amostras")
ax.set_title("Balanceamento das classes")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.savefig("../results/04_balanceamento_classes.png", bbox_inches="tight")
plt.show()


**Leitura:** apenas **~13,9%** das amostras são de Alta Qualidade — um desbalanceamento relevante. Isso tem implicações diretas na modelagem:

- Acurácia sozinha é uma métrica enganosa aqui (um modelo "bobo" que sempre prevê "Baixa/Média" já acertaria ~86% das vezes, mas seria inútil na prática).
- Vamos priorizar **F1-score, recall, precisão e ROC-AUC** na avaliação.
- Usaremos `class_weight="balanced"` nos modelos para compensar o desbalanceamento sem precisar descartar dados ou gerar amostras sintéticas.

### 2.6 Variáveis mais discriminantes vs. classe alvo

In [ ]:
top_feats = corr["quality"].drop("quality").abs().sort_values(ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(top_feats):
    sns.boxplot(x="high_quality", y=col, data=df, ax=axes[i], palette=["#7B2D26", "#C9A227"])
    axes[i].set_xticklabels(["Baixa/Média", "Alta"])
    axes[i].set_xlabel("")
plt.tight_layout()
plt.savefig("../results/05_top_features_vs_target.png", bbox_inches="tight")
plt.show()


**Leitura:** os boxplots confirmam visualmente o que a correlação já indicava — vinhos de Alta Qualidade têm mediana de `alcohol` visivelmente maior e de `volatile acidity` visivelmente menor, com pouca sobreposição entre as distribuições. Isso sugere que mesmo um modelo simples já capturaria boa parte do sinal só com essas duas variáveis, mas a combinação com as demais deve refinar a fronteira de decisão.

## 3. Pré-processamento de Dados

### 3.1 Tratamento de outliers

**Método escolhido: capping (winsorização) via IQR**, aplicando `clip` nos limites `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.

**Por que capping e não remoção de linhas?**
- O dataset já é pequeno (1.143 amostras) e a classe de interesse (Alta Qualidade) já é minoritária (~13,9%) — remover linhas arriscaria eliminar justamente vinhos atípicos que podem estar entre os de nota mais alta.
- Os "outliers" identificados são fisicamente plausíveis (não são erros de digitação ou de sensor, como um pH negativo) — são apenas amostras mais extremas dentro do processo real de vinificação.
- O capping reduz a influência desproporcional de valores extremos no treinamento (especialmente relevante para o SVM, sensível a escala) sem descartar informação da amostra.

In [ ]:
df_clean = df.copy()

for col in feature_cols:
    q1, q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df_clean[col] = df_clean[col].clip(lower=low, upper=high)

df_clean[feature_cols].describe().T[["min", "max"]]


### 3.2 Feature Engineering

Criamos 4 variáveis derivadas com justificativa de domínio (enologia):

| Feature | Fórmula | Justificativa |
|---|---|---|
| `acid_balance` | `fixed acidity / volatile acidity` | Equilíbrio entre acidez "boa" (estrutural) e acidez "ruim" (defeito de vinagre); a razão captura melhor esse equilíbrio do que as variáveis isoladas |
| `free_sulfur_ratio` | `free SO2 / total SO2` | Fração do SO2 que permanece ativa como conservante/antioxidante — mais informativa que os valores absolutos |
| `alcohol_density` | `alcohol / density` | Álcool e densidade são fisicamente relacionados (mais álcool → menor densidade); a razão reforça esse efeito conjunto |
| `acidity_alcohol_interaction` | `fixed acidity * alcohol` | Interação entre dois dos fatores mais associados a boas notas |

In [ ]:
df_clean["acid_balance"] = df_clean["fixed acidity"] / (df_clean["volatile acidity"] + 1e-6)
df_clean["free_sulfur_ratio"] = df_clean["free sulfur dioxide"] / (df_clean["total sulfur dioxide"] + 1e-6)
df_clean["alcohol_density"] = df_clean["alcohol"] / df_clean["density"]
df_clean["acidity_alcohol_interaction"] = df_clean["fixed acidity"] * df_clean["alcohol"]

engineered_feats = ["acid_balance", "free_sulfur_ratio", "alcohol_density", "acidity_alcohol_interaction"]
all_feature_cols = feature_cols + engineered_feats

df_clean[engineered_feats].describe().T


### 3.3 Separação treino/teste e escalonamento

- **Split estratificado 80/20** — mantém a mesma proporção de classes em treino e teste, essencial dado o desbalanceamento.
- **StandardScaler** aplicado às variáveis numéricas — necessário para o SVM (sensível à escala das features, pois depende de distância/produto interno); o Random Forest é invariante a escala, mas usar os mesmos dados escalonados não prejudica seu desempenho, então mantemos os dois conjuntos disponíveis.

In [ ]:
X = df_clean[all_feature_cols]
y = df_clean["high_quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("Proporção classe Alta - treino:", y_train.mean().round(3), "| teste:", y_test.mean().round(3))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


## 4. Desenvolvimento de Modelos

Treinamos dois modelos de classificação bem diferentes em natureza, para uma comparação rica:

- **Random Forest:** ensemble de árvores de decisão, robusto a outliers e não-linearidades, não exige escalonamento e fornece importância de variáveis nativamente.
- **SVM (kernel RBF):** busca a fronteira de decisão de máxima margem em um espaço transformado, mais sensível a escala mas capaz de capturar fronteiras não-lineares complexas.

Ambos usam `class_weight="balanced"` para lidar com o desbalanceamento das classes, e têm hiperparâmetros otimizados via **GridSearchCV com validação cruzada estratificada (5 folds)**, otimizando F1-score (métrica mais adequada que acurácia neste contexto).

### 4.1 Random Forest

In [ ]:
rf_param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 8, 12],
    "min_samples_leaf": [1, 3],
}
rf_base = RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)
rf_grid = GridSearchCV(rf_base, rf_param_grid, scoring="f1", cv=cv, n_jobs=-1)
rf_grid.fit(X_train, y_train)
rf_best = rf_grid.best_estimator_

print("Melhores hiperparâmetros (RF):", rf_grid.best_params_)
print("Melhor F1 (validação cruzada):", round(rf_grid.best_score_, 4))


### 4.2 SVM (kernel RBF)

In [ ]:
svm_param_grid = {
    "C": [1, 5, 10],
    "gamma": ["scale", 0.01, 0.1],
    "kernel": ["rbf"],
}
svm_base = SVC(class_weight="balanced", probability=True, random_state=RANDOM_STATE)
svm_grid = GridSearchCV(svm_base, svm_param_grid, scoring="f1", cv=cv, n_jobs=-1)
svm_grid.fit(X_train_scaled, y_train)
svm_best = svm_grid.best_estimator_

print("Melhores hiperparâmetros (SVM):", svm_grid.best_params_)
print("Melhor F1 (validação cruzada):", round(svm_grid.best_score_, 4))


## 5. Avaliação dos Modelos

Métricas usadas — todas relevantes para um problema binário desbalanceado:

- **Acurácia:** taxa geral de acerto (usada com cautela, dado o desbalanceamento)
- **Precisão:** dos vinhos previstos como Alta Qualidade, quantos realmente são
- **Recall (sensibilidade):** dos vinhos realmente de Alta Qualidade, quantos o modelo conseguiu identificar
- **F1-score:** média harmônica entre precisão e recall
- **ROC-AUC:** capacidade de separar as classes considerando todos os limiares de decisão
- **Validação cruzada (5 folds):** para checar a estabilidade das métricas além do split único de teste

In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    cv_scores = cross_validate(
        model, X_tr, y_tr, cv=cv,
        scoring=["accuracy", "precision", "recall", "f1", "roc_auc"]
    )

    metrics = {
        "accuracy": accuracy_score(y_te, y_pred),
        "precision": precision_score(y_te, y_pred),
        "recall": recall_score(y_te, y_pred),
        "f1": f1_score(y_te, y_pred),
        "roc_auc": roc_auc_score(y_te, y_proba),
        "cv_f1_mean": cv_scores["test_f1"].mean(),
        "cv_f1_std": cv_scores["test_f1"].std(),
        "cv_roc_auc_mean": cv_scores["test_roc_auc"].mean(),
    }
    cm = confusion_matrix(y_te, y_pred)
    report = classification_report(y_te, y_pred, target_names=["Baixa/Média", "Alta"])
    return metrics, cm, report, y_pred, y_proba

rf_metrics, rf_cm, rf_report, rf_pred, rf_proba = evaluate(rf_best, X_train, y_train, X_test, y_test)
svm_metrics, svm_cm, svm_report, svm_pred, svm_proba = evaluate(svm_best, X_train_scaled, y_train, X_test_scaled, y_test)

metrics_df = pd.DataFrame({
    "Random Forest": rf_metrics,
    "SVM (RBF)": svm_metrics,
}).T
metrics_df.round(4)


In [ ]:
print("=== Random Forest ===")
print(rf_report)
print("=== SVM (RBF) ===")
print(svm_report)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, cm, title in zip(axes, [rf_cm, svm_cm], ["Random Forest", "SVM (RBF)"]):
    sns.heatmap(cm, annot=True, fmt="d", cmap="RdBu_r", cbar=False, ax=ax,
                xticklabels=["Baixa/Média", "Alta"], yticklabels=["Baixa/Média", "Alta"])
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
    ax.set_title(title)
plt.tight_layout()
plt.savefig("../results/06_matrizes_confusao.png", bbox_inches="tight")
plt.show()


In [ ]:
plt.figure(figsize=(6.5, 5.5))
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_proba)
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={rf_metrics['roc_auc']:.3f})", color="#7B2D26", lw=2)
plt.plot(fpr_svm, tpr_svm, label=f"SVM (AUC={svm_metrics['roc_auc']:.3f})", color="#C9A227", lw=2)
plt.plot([0, 1], [0, 1], linestyle="--", color="grey", lw=1)
plt.xlabel("Taxa de Falso Positivo")
plt.ylabel("Taxa de Verdadeiro Positivo")
plt.title("Curvas ROC - Comparação dos modelos")
plt.legend()
plt.tight_layout()
plt.savefig("../results/07_curvas_roc.png", bbox_inches="tight")
plt.show()


In [ ]:
metrics_plot = metrics_df[["accuracy", "precision", "recall", "f1", "roc_auc"]]
metrics_plot.plot(kind="bar", figsize=(9, 5), color=["#7B2D26", "#A8442A", "#C9752E", "#C9A227", "#8C8C56"])
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Comparação de métricas entre modelos")
plt.legend(loc="lower right", ncol=3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../results/08_comparacao_metricas.png", bbox_inches="tight")
plt.show()


**Leitura comparativa:**

- O **Random Forest supera o SVM em praticamente todas as métricas** no conjunto de teste, com destaque para ROC-AUC e F1-score — indicando melhor capacidade de separar as duas classes e melhor equilíbrio entre precisão e recall.
- Ambos os modelos têm recall mais baixo que a precisão, refletindo o desafio real do desbalanceamento: identificar todos os vinhos de Alta Qualidade é mais difícil do que confirmar os que o modelo já aponta como tal.
- A validação cruzada confirma que o desempenho do Random Forest é consistente (baixo desvio padrão entre os folds), reduzindo a chance de que o resultado no teste tenha sido "sorte" do split.
- Isso é consistente com a natureza dos dados: Random Forest lida naturalmente melhor com relações não-lineares e interações entre variáveis físico-químicas, além de ser mais robusto a outliers residuais e à multicolinearidade observada na correlação (ex: `fixed acidity` x `citric acid`).

## 6. Interpretação dos Resultados

### 6.1 Importância das variáveis

- **Random Forest:** importância nativa (baseada na redução média de impureza Gini)
- **SVM:** importância por **permutação** (mede a queda de desempenho ao embaralhar cada variável) — necessária porque o SVM com kernel RBF não tem coeficientes diretamente interpretáveis como um modelo linear

In [ ]:
rf_importances = pd.Series(rf_best.feature_importances_, index=all_feature_cols).sort_values(ascending=False)

perm = permutation_importance(svm_best, X_test_scaled, y_test, n_repeats=20, random_state=RANDOM_STATE, scoring="f1")
svm_importances = pd.Series(perm.importances_mean, index=all_feature_cols).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
rf_importances.head(10).sort_values().plot(kind="barh", ax=axes[0], color="#7B2D26")
axes[0].set_title("Random Forest - Importância (Gini)")
svm_importances.head(10).sort_values().plot(kind="barh", ax=axes[1], color="#C9A227")
axes[1].set_title("SVM - Importância por permutação")
plt.tight_layout()
plt.savefig("../results/09_importancia_variaveis.png", bbox_inches="tight")
plt.show()


### 6.2 Discussão e implicações para o processo produtivo

**Variáveis mais influentes (consistentes nos dois modelos):**

1. **`alcohol` (teor alcoólico):** o preditor mais forte em ambos os modelos. Na prática, isso reforça o vínculo entre maturação da uva na colheita (que eleva o teor de açúcar convertido em álcool na fermentação) e a qualidade percebida — um indicativo de que decisões sobre o ponto de colheita têm impacto direto na qualidade final.
2. **`volatile acidity` (acidez volátil):** segundo fator mais relevante, com relação inversa. Controlar a fermentação e evitar contaminação bacteriana (principal causa de acidez volátil elevada) é uma alavanca operacional direta para reduzir a proporção de vinhos de baixa qualidade.
3. **`sulphates` e as features derivadas de enxofre (`free_sulfur_ratio`):** reforçam que o manejo da dosagem de conservantes (SO2) tem um papel mensurável na qualidade final, não apenas na conservação.
4. As **variáveis criadas na engenharia de features** (`acid_balance`, `alcohol_density`) aparecem com importância relevante em pelo menos um dos modelos, sugerindo que capturar *relações* entre variáveis (e não só seus valores isolados) agrega poder preditivo.

**Implicações práticas:**

- Um modelo como o Random Forest pode ser usado como **triagem preliminar** em lotes de produção, sinalizando amostras com alta probabilidade de nota baixa antes mesmo da avaliação sensorial — otimizando o tempo dos enólogos.
- Como o recall ainda é a métrica mais desafiadora, o modelo é mais confiável hoje para **confirmar** vinhos de alta qualidade do que para garantir que nenhum seja perdido — importante deixar isso claro para quem for usar o modelo na prática.
- Investir em maior volume de dados (principalmente mais amostras de vinhos nota 7–8) tende a ser a alavanca mais eficaz para melhorar o recall em versões futuras do modelo.

## Conclusão

Random Forest foi o modelo mais indicado para este problema, com melhor desempenho em praticamente todas as métricas relevantes e maior estabilidade em validação cruzada. Ainda assim, o desbalanceamento de classes limita o recall de ambos os modelos — uma limitação relevante para qualquer aplicação prática que dependa de captar corretamente todos os vinhos de alta qualidade.

Próximos passos sugeridos: testar técnicas de balanceamento como SMOTE, ampliar a base de dados (principalmente a classe minoritária) e explorar modelos adicionais (ex: Gradient Boosting) para validar se o ganho de desempenho compensa a maior complexidade.